In [ ]:
!pip install pyspark

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving attendance.csv to attendance (1).csv
Saving employees.csv to employees (1).csv
Saving tasks.csv to tasks (1).csv


In [22]:
from pyspark.sql import SparkSession
# =========================================
# CREATE SPARK SESSION
# =========================================

spark = SparkSession.builder \
    .appName("Employee Attendance Analysis") \
    .getOrCreate()
print("Spark session created...!")

Spark session created...!


In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

# =========================================
# LOAD CSV FILES
# =========================================

employees = spark.read.csv(
    "employees.csv",
    header=True,
    inferSchema=True
)

attendance = spark.read.csv(
    "attendance.csv",
    header=True,
    inferSchema=True
)

tasks = spark.read.csv(
    "tasks.csv",
    header=True,
    inferSchema=True
)


In [ ]:
# =========================================
# SHOW DATA
# =========================================

print("Employees Data")
employees.show()

print("Attendance Data")
attendance.show()

print("Tasks Data")
tasks.show()


Employees Data
+-----------+-------------+----------+------------+-----------------+
|employee_id|employee_name|department|joining_date|            email|
+-----------+-------------+----------+------------+-----------------+
|        101|        Arjun|        HR|  2023-01-12|  arjun@gmail.com|
|        102|        Priya|        IT|  2022-09-18|  priya@gmail.com|
|        103|        Rahul|   Finance|  2021-11-05|  rahul@gmail.com|
|        104|        Sneha| Marketing|  2023-03-21|  sneha@gmail.com|
|        105|       Vikram|        IT|  2020-07-15| vikram@gmail.com|
|        106|        Meena|        HR|  2022-05-11|  meena@gmail.com|
|        107|        Karan|     Sales|  2021-08-30|  karan@gmail.com|
|        108|        Divya|   Finance|  2023-06-14|  divya@gmail.com|
|        109|         Ajay| Marketing|  2022-10-02|   ajay@gmail.com|
|        110|        Nisha|     Sales|  2021-12-19|  nisha@gmail.com|
|        111|        Rohit|        IT|  2020-04-25|  rohit@gmail.com|
|    

In [ ]:
# =========================================
# FILTER LATE EMPLOYEES
# =========================================

late_employees = attendance.filter(
    col("status") == "Late"
)

print("Late Employees")
late_employees.show()


Late Employees
+-------------+-----------+----------+-------------------+-------------------+------+
|attendance_id|employee_id|      date|           clock_in|          clock_out|status|
+-------------+-----------+----------+-------------------+-------------------+------+
|            2|        102|2026-05-01|2026-05-13 09:45:00|2026-05-13 18:10:00|  Late|
|            5|        105|2026-05-01|2026-05-13 09:50:00|2026-05-13 18:30:00|  Late|
|            9|        109|2026-05-01|2026-05-13 09:35:00|2026-05-13 18:40:00|  Late|
|           12|        112|2026-05-02|2026-05-13 09:55:00|2026-05-13 18:25:00|  Late|
|           15|        115|2026-05-02|2026-05-13 09:40:00|2026-05-13 18:20:00|  Late|
|           19|        119|2026-05-02|2026-05-13 09:50:00|2026-05-13 18:35:00|  Late|
+-------------+-----------+----------+-------------------+-------------------+------+



In [ ]:
# =========================================
# FILTER ABSENT EMPLOYEES
# =========================================

absent_employees = attendance.filter(
    col("status") == "Absent"
)

print("Absent Employees")
absent_employees.show()


Absent Employees
+-------------+-----------+----------+-------------------+-------------------+------+
|attendance_id|employee_id|      date|           clock_in|          clock_out|status|
+-------------+-----------+----------+-------------------+-------------------+------+
|            3|        103|2026-05-01|2026-05-13 00:00:00|2026-05-13 00:00:00|Absent|
|            7|        107|2026-05-01|2026-05-13 00:00:00|2026-05-13 00:00:00|Absent|
|           13|        113|2026-05-02|2026-05-13 00:00:00|2026-05-13 00:00:00|Absent|
|           17|        117|2026-05-02|2026-05-13 00:00:00|2026-05-13 00:00:00|Absent|
+-------------+-----------+----------+-------------------+-------------------+------+



In [ ]:
# =========================================
# MERGE DATASETS
# =========================================

merged_data = employees.join(
    attendance,
    "employee_id"
).join(
    tasks,
    "employee_id"
)


In [ ]:
# =========================================
# CALCULATE PRODUCTIVITY
# =========================================

merged_data = merged_data.withColumn(
    "productivity_score",
    col("tasks_completed") * 10
)


In [ ]:
# =========================================
# GROUP BY DEPARTMENT
# =========================================

department_analysis = merged_data.groupBy(
    "department"
).agg(
    avg("hours_spent").alias("avg_work_hours"),
    avg("productivity_score").alias("avg_productivity")
)

print("Department-wise Analysis")
department_analysis.show()


Department-wise Analysis
+----------+--------------+----------------+
|department|avg_work_hours|avg_productivity|
+----------+--------------+----------------+
|     Sales|           5.5|            40.0|
|        HR|           5.0|            37.5|
|   Finance|           3.5|            25.0|
| Marketing|          6.75|            50.0|
|        IT|           6.5|            45.0|
+----------+--------------+----------------+



In [ ]:
# =========================================
# ATTENDANCE ISSUES REPORT
# =========================================

attendance_issues = attendance.groupBy(
    "status"
).count()

print("Attendance Issues")
attendance_issues.show()

Attendance Issues
+-------+-----+
| status|count|
+-------+-----+
|Present|   10|
| Absent|    4|
|   Late|    6|
+-------+-----+

